In [96]:
import os
from pathlib import Path
import re

import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from Bio import Phylo

cwd = os.getcwd()
if cwd.endswith('bacteriocins'):
    os.chdir('../..')
    cwd = os.getcwd()

from src.tree.itol_annotation import itol_heatmap_annotations

In [97]:
sns.set_palette('colorblind')
sns.set_style('whitegrid')
sns.set_context('paper', font_scale=1.8)
plt.rcParams['font.family'] = 'Helvetica'

palette = sns.color_palette().as_hex()

data_folder = Path('data/amp_db')
assert data_folder.is_dir()

outputs_folder = Path('data/outputs/amp_db')


## Load data

In [98]:
gtdb_metadata = pd.read_csv(data_folder / 'gtdb_metadata.csv.gz', index_col='ncbi_accession')

In [99]:
bacteriocins_df = pd.read_csv(data_folder / 'bacteriocins.csv.gz')
bacteriocins_df = bacteriocins_df[
    (bacteriocins_df['amp_id'] != 'pg_hydrolase') &
    (bacteriocins_df['bitscore'] >= 20)
].copy()
bacteriocins_df.head()

,assembly_accession,protein_id,amp_id,amp_desc,evalue,bitscore,ncbi_accession,domain,gtdb_phylum,gtdb_class,gtdb_order,gtdb_family,gtdb_genus,gtdb_species,ncbi_organism_name
5,GCA_000010565.1,BAF60466.1,PoyA,NaN,1.389000e-12,61.0,GCA_000010565.1,Bacteria,Bacillota_B,Desulfotomaculia,Desulfotomaculales,Pelotomaculaceae,Pelotomaculum,Pelotomaculum thermopropionicum,Pelotomaculum thermopropionicum SI
6,GCA_000010565.1,BAF60467.1,PoyA,NaN,2.750000e-10,55.0,GCA_000010565.1,Bacteria,Bacillota_B,Desulfotomaculia,Desulfotomaculales,Pelotomaculaceae,Pelotomaculum,Pelotomaculum thermopropionicum,Pelotomaculum thermopropionicum SI
7,GCA_000010565.1,BAF60468.1,PoyA,NaN,1.296000e-09,53.0,GCA_000010565.1,Bacteria,Bacillota_B,Desulfotomaculia,Desulfotomaculales,Pelotomaculaceae,Pelotomaculum,Pelotomaculum thermopropionicum,Pelotomaculum thermopropionicum SI
8,GCA_000010565.1,BAF60489.1,Bacteriocin_IIc,PF10439,9.300000e-08,29.5,GCA_000010565.1,Bacteria,Bacillota_B,Desulfotomaculia,Desulfotomaculales,Pelotomaculaceae,Pelotomaculum,Pelotomaculum thermopropionicum,Pelotomaculum thermopropionicum SI
9,GCA_000010565.1,BAF60509.1,PoyA,NaN,3.282000e-09,51.0,GCA_000010565.1,Bacteria,Bacillota_B,Desulfotomaculia,Desulfotomaculales,Pelotomaculaceae,Pelotomaculum,Pelotomaculum thermopropionicum,Pelotomaculum thermopropionicum SI


In [100]:
bacteriocins_df[['domain', 'assembly_accession']].groupby('domain').count()

,assembly_accession
domain,
Archaea,327
Bacteria,10577


## Assign phylum group

In [101]:
def is_named_phylum(phylum):
    if re.match(r'^[A-Z0-9\-_]+$', phylum) is not None:
        return False

    blacklist = {
        'BMS3Abin14',
        'BS750m-G25',
        'BS750m-G34',
        'JdFR-76',
        'T1Sed10-126',
        'SpSt-1190',
        'B1Sed10-29',
        'EX4484-52',
    }
    return phylum not in blacklist


def get_phylum_group(phylum):
    m = re.match(r'^(.+)_[A-Z]$', phylum)
    if m is not None:
        return m[1]
    else:
        return phylum


named_phyla = [
    p for p in gtdb_metadata['gtdb_phylum'].unique()
    if is_named_phylum(p)
]
map_phylm_groups = {
    phylum: get_phylum_group(phylum)
    for phylum in named_phyla
}
bacteriocins_df['phylum_group'] = bacteriocins_df['gtdb_phylum'].apply(lambda p: map_phylm_groups.get(p))

## Bacteriocins present in both domains

In [102]:
arc_bacteriocins = set(bacteriocins_df[bacteriocins_df['domain'] == 'Archaea']['amp_id'].unique())
bac_bacteriocins = set(bacteriocins_df[bacteriocins_df['domain'] == 'Bacteria']['amp_id'].unique())

shared_bacteriocins = sorted(arc_bacteriocins & bac_bacteriocins)

print(f'Number of bacteriocins shared between archaea and bacteria: {len(shared_bacteriocins):,}')

Number of bacteriocins shared between archaea and bacteria: 85


In [103]:
display_name_map = {
    'Competence': 'ComX homolog',
    'ComX2': 'ComX homolog',
    'ComX4': 'ComX homolog',
    'PaeM': 'Colicin M',
    'Gardimycin_(actagardine)': 'Actagardine',
    'Lactococcin_Q_chain_a': 'Lactococcin Q (alpha)',
    'Lactococcin_Q_chain_b': 'Lactococcin Q (beta)',
    'Enterocin_NKR_5_3D': 'Enterocin NKR-5-3D',
    'EnterocinP': 'Enterocin P',
    'Subtilosin_(SboX)': 'Subtilosin',
    'GarvieacinQ garQ': 'Garvieacin Q',
    'Lactococcin': 'Lactococcin A',
    'Lactococcin_A_(LCN_A)': 'Lactococcin A',
    'PoyA': 'Polytheonamide B',
}
def set_display_name(name):
    if name in display_name_map:
        return display_name_map[name]
    else:
        return name.replace('_', ' ')
    
bacteriocins_df['display_name'] = bacteriocins_df['amp_id'].apply(set_display_name)

amp_id_to_diplay_name = {
    tpl.amp_id: tpl.display_name
    for tpl in bacteriocins_df.itertuples()
}

Putative bacteriocins excluded from the heatmap:

- `Linocin M18` is shown to have genuine use for harvesting iron in Pyrococcus furiosus: https://www.uniprot.org/uniprotkb/Q8U1L4/entry
- `FlvA2h` and `FlvA2f` are part of a `Lantipeptide` in _Ruminococcus flavefaciens_ but these two genes are not the active part as per: https://www.uniprot.org/uniprotkb/P0DQM0/entry

Interesting but not sure if we should include:
- `Competence`, `ComX2` and `ComX4` are both `comX` in _B. subtilis_. mimic of comX can be used to mess with quorum sensing: https://doi.org/10.1016/j.pbi.2004.05.008
- Same story but less clearcut for `Auto_Inducing_Peptide_III` which map to TIGR entry TIGR04223: https://www.ncbi.nlm.nih.gov/genome/annotation_prok/evidence/TIGR04223/

Name change:
- `PaeM` is better known as `Colicin M` as per automatic Pfam annotations: https://www.uniprot.org/uniprotkb/Q88A25/entry
- `Gardimycin_(actagardine)` renamed `Actagardine`
- `Lactococcin_A_(LCN_A)` and `Lactococcin` are renamed `Lactococcin A`.

In [104]:
bacteriocins_to_exclude = {
    'Auto_Inducing_Peptide_III',
    'Linocin_M18',
    'FlvA2f',
    'FlvA2h',
    'Competence',
    'ComX2',
    'ComX4',
    'Enterocin_NKR_5_3D',
    'Lactococcin_Q_chain_a',
    'Lactococcin_Q_chain_b',
    'Piricyclamide_7005E2',
    'Enterocin_1071A',
    'Enterocin_1071B',
    'SSV_2083',
    'Caulonodin_III',
    'Ancovenin',
}
bacteriocins_shortlist = set({
    amp_id_to_diplay_name[amp_id]
    for amp_id in shared_bacteriocins
    if amp_id not in bacteriocins_to_exclude
})
len(bacteriocins_shortlist)

66

In [105]:
counts_per_amp = bacteriocins_df[
    bacteriocins_df['display_name'].isin(bacteriocins_shortlist)
][
    ['display_name', 'assembly_accession']
].groupby('display_name').nunique().rename(columns={'assembly_accession': 'n_assemblies'})

counts_per_domain = bacteriocins_df[
    ['domain', 'display_name', 'assembly_accession'
]].groupby(['domain', 'display_name']).nunique()

archaea_counts = counts_per_domain.loc['Archaea'].copy().rename(columns={'assembly_accession': 'n_archaea'})
bacteria_counts = counts_per_domain.loc['Bacteria'].copy().rename(columns={'assembly_accession': 'n_bacteria'})

def compute_clade_count(df, amp_id, domain, clade='gtdb_class'):
    return len(df[
        (df['domain'] == domain) &
        (df['display_name'] == amp_id)
    ][clade].unique())

amp_table = pd.merge(
    counts_per_amp,
    archaea_counts,
    how='left',
    on='display_name',
)
amp_table = pd.merge(
    amp_table,
    bacteria_counts,
    how='left',
    on='display_name',
)
amp_table['percent_archaea'] = (
    100 * amp_table['n_archaea'] / len(gtdb_metadata[gtdb_metadata['domain'] == 'Archaea'])
).round(2)
amp_table['percent_bacteria'] = (
    100 * amp_table['n_bacteria'] / len(gtdb_metadata[gtdb_metadata['domain'] == 'Bacteria'])
).round(2)
amp_table['percent_diff'] = (amp_table['percent_bacteria'] - amp_table['percent_archaea']).round(2)
amp_table['n_archaeal_phyla'] = [
    compute_clade_count(bacteriocins_df, amp_id, domain='Archaea', clade='gtdb_phylum')
    for amp_id in amp_table.index
]
amp_table['n_bacterial_phyla'] = [
    compute_clade_count(bacteriocins_df, amp_id, domain='Bacteria', clade='gtdb_phylum')
    for amp_id in amp_table.index
]
amp_table['n_archaeal_classes'] = [
    compute_clade_count(bacteriocins_df, amp_id, domain='Archaea', clade='gtdb_class')
    for amp_id in amp_table.index
]
amp_table['n_bacterial_classes'] = [
    compute_clade_count(bacteriocins_df, amp_id, domain='Bacteria', clade='gtdb_class')
    for amp_id in amp_table.index
]

# Selection criteria: at least 2 genomes in each domain.
amp_table = amp_table[
    (amp_table['n_archaea'] >= 2)
    & (amp_table['n_bacteria'] >= 2)
].copy()

amp_table = amp_table.sort_values(['n_archaea'], ascending=False)

print(len(amp_table),amp_table['n_archaea'].sum())

amp_table

15 45


,n_assemblies,n_archaea,n_bacteria,percent_archaea,percent_bacteria,percent_diff,n_archaeal_phyla,n_bacterial_phyla,n_archaeal_classes,n_bacterial_classes
display_name,,,,,,,,,,
Lactococcin A,163,8,155,0.22,0.31,0.09,7,26,8,42
Nocardithiocin,43,5,38,0.13,0.08,-0.05,2,11,2,16
Enterocin P,37,4,33,0.11,0.07,-0.04,3,11,4,16
Lactococcin MMFII,12,4,8,0.11,0.02,-0.09,3,5,3,6
Colicin M,50,3,47,0.08,0.09,0.01,3,12,3,16
Sakacin A,14,3,11,0.08,0.02,-0.06,3,9,3,10
Actagardine,15,2,13,0.05,0.03,-0.02,2,4,2,5
Aureocin A53,6,2,4,0.05,0.01,-0.04,1,4,1,4
Bacteriocin 31,40,2,38,0.05,0.08,0.03,2,11,2,14


In [106]:
df = bacteriocins_df[
    (bacteriocins_df['domain'] == 'Archaea') &
    bacteriocins_df['display_name'].isin(amp_table.index)
].sort_values(['bitscore', 'evalue'], ascending=[False, True])

print(len(df), len(df['display_name'].unique()))

45 15


In [123]:
df[[
    'display_name', 'assembly_accession', 'protein_id', 'evalue', 'bitscore', 
    'domain', 'gtdb_phylum', 'gtdb_class', 'gtdb_order', 'gtdb_family', 'gtdb_genus', 'gtdb_species', 'ncbi_organism_name',
]].rename(columns={
    'display_name': 'bacteriocin',
}).sort_values([
    'bacteriocin', 'gtdb_phylum', 'gtdb_class', 'gtdb_order', 'gtdb_family', 'gtdb_genus', 'gtdb_species', 'ncbi_organism_name',
]).to_csv('/Users/rs1521/Documents/00_Thesis/tables/bacteriocins_in_archaea.csv', index=False)

In [108]:
df[['display_name', 'bitscore']].groupby('display_name').mean().sort_values('bitscore', ascending=False)

,bitscore
display_name,
Subtilosin A,51.500000
Polytheonamide B,39.500000
Colicin M,39.000000
Actagardine,34.500000
Microcin E492,34.000000
Hiracin JM79,33.500000
Lactococcin A,33.087500
Aureocin A53,33.000000
Bacteriocin 31,33.000000


In [109]:
sorted(df['display_name'].unique())

['Actagardine',
 'Aureocin A53',
 'Bacteriocin 31',
 'Colicin M',
 'Enterocin L50b',
 'Enterocin P',
 'Garvicin Q',
 'Hiracin JM79',
 'Lactococcin A',
 'Lactococcin MMFII',
 'Microcin E492',
 'Nocardithiocin',
 'Polytheonamide B',
 'Sakacin A',
 'Subtilosin A']

In [110]:
metadata = gtdb_metadata.loc[
    df['assembly_accession'].values
][
    ['gtdb_species', 'ncbi_organism_name', 'checkm_completeness', 'checkm_contamination', 'ncbi_assembly_level',]
].copy()
metadata['is_complete'] = metadata['ncbi_assembly_level'] == 'Complete Genome'
metadata.sort_values(
    ['is_complete', 'checkm_completeness', 'checkm_contamination'],
    ascending=[False, False, True],
)

,gtdb_species,ncbi_organism_name,checkm_completeness,checkm_contamination,ncbi_assembly_level,is_complete
ncbi_accession,,,,,,
GCF_002214365.1,Thermococcus celer,Thermococcus celer Vu 13 = JCM 8558,100.00,0.00,Complete Genome,True
GCF_000585495.1,Thermococcus nautili,Thermococcus nautili,100.00,0.00,Complete Genome,True
GCF_000970205.1,Methanosarcina mazei,Methanosarcina mazei S-6,100.00,0.65,Complete Genome,True
GCF_000969965.1,Methanosarcina sp000969965,Methanosarcina sp. WWM596,99.84,0.00,Complete Genome,True
GCF_000970045.1,MTP4 sp000970045,Methanosarcina sp. MTP4,99.84,0.00,Complete Genome,True
GCF_014647415.1,Haloarcula sebkhae,Haloarcula sebkhae,99.93,0.40,Scaffold,False
GCF_000383975.1,Natronorubrum tibetense,Natronorubrum tibetense GA33,99.36,0.03,Scaffold,False
GCF_900100335.1,Natronorubrum texcoconense,Natronorubrum texcoconense,99.36,1.14,Contig,False
GCA_012026835.1,Methanoperedens sp012026835,Candidatus Methanoperedens sp.,99.35,4.58,Contig,False


## Bacteria only bacteriocins subset

In [111]:
bacteria_only_shortlist = [
    'Lactococcin 972',
    'Closticin 574',
    'Colicin E6',
    'Colicin E9',
]

## Make heatmap

In [112]:
bacteriocins_to_plot = [
    amp_display_name
    for amp_display_name in amp_table.index
] + bacteria_only_shortlist

print(f'Number of bacteriocins: {len(bacteriocins_to_plot)}')

bacteriocins_to_plot

Number of bacteriocins: 19


['Lactococcin A',
 'Nocardithiocin',
 'Enterocin P',
 'Lactococcin MMFII',
 'Colicin M',
 'Sakacin A',
 'Actagardine',
 'Aureocin A53',
 'Bacteriocin 31',
 'Enterocin L50b',
 'Garvicin Q',
 'Hiracin JM79',
 'Microcin E492',
 'Polytheonamide B',
 'Subtilosin A',
 'Lactococcin 972',
 'Closticin 574',
 'Colicin E6',
 'Colicin E9']

In [113]:
def compute_heatmap_data(
    gtdb_metadata, 
    bacteriocins_df, 
    phylum_groups, 
    bacteriocin_names, 
):
    gtdb_metadata['phylum_group'] = gtdb_metadata['gtdb_phylum'].apply(lambda p: map_phylm_groups.get(p))
    metadata = gtdb_metadata[gtdb_metadata['phylum_group'].isin(phylum_groups)].reset_index()
    phylum_count_df = metadata[
        ['phylum_group', 'ncbi_accession']
    ].groupby('phylum_group').count().rename(columns={
        'ncbi_accession': 'n_genomes'
    })
    phylum_count = {
        phylum: phylum_count_df.loc[phylum, 'n_genomes']
        for phylum in phylum_groups
    }

    heatmap_data = {
        'phylum_group': [],
        'n_assemblies': [],
    }
    for bacteriocin in bacteriocin_names:
        heatmap_data[bacteriocin] = []

    for phylum in phylum_groups:
        bacteriocin_count = bacteriocins_df[
            bacteriocins_df['display_name'].isin(bacteriocin_names) &
            (bacteriocins_df['phylum_group'] == phylum)
        ][['display_name', 'ncbi_accession']].groupby('display_name').nunique().rename(columns={
            'ncbi_accession': 'n_genomes'
        })

        heatmap_data['phylum_group'].append(phylum)
        heatmap_data['n_assemblies'].append(phylum_count[phylum])
        for bacteriocin in bacteriocin_names:
            if bacteriocin in bacteriocin_count.index:
                count = bacteriocin_count.loc[bacteriocin, 'n_genomes']
            else:
                count = 0
            
            heatmap_data[bacteriocin].append(count)

    return pd.DataFrame.from_dict(heatmap_data).set_index('phylum_group', drop=True)

In [114]:
arc_phylum_group_tree = Phylo.read(Path('data/archaea-vs-bacteria/figure1/ar53_phylum_groups.tree'), 'phyloxml')
bac_phylum_group_tree = Phylo.read(Path('data/archaea-vs-bacteria/figure1/bac120_phylum_groups.tree'), 'phyloxml')

arc_phylum_groups = sorted([l.name for l in arc_phylum_group_tree.get_terminals()])
bac_phylum_groups = sorted([l.name for l in bac_phylum_group_tree.get_terminals()])
all_phylum_groups = sorted(set(arc_phylum_groups) | set(bac_phylum_groups))

print(f'Number of archaeal phylum groups: {len(arc_phylum_groups):,}')
print(f'Number of bacterial phylum groups: {len(bac_phylum_groups):,}')

Number of archaeal phylum groups: 14
Number of bacterial phylum groups: 34


In [115]:
archaeal_heatmap_df = compute_heatmap_data(
    gtdb_metadata, 
    bacteriocins_df[bacteriocins_df['domain'] == 'Archaea'].copy(), 
    phylum_groups=arc_phylum_groups, 
    bacteriocin_names=bacteriocins_to_plot, 
)
archaeal_heatmap_df

,n_assemblies,Lactococcin A,Nocardithiocin,Enterocin P,Lactococcin MMFII,Colicin M,Sakacin A,Actagardine,Aureocin A53,Bacteriocin 31,Enterocin L50b,Garvicin Q,Hiracin JM79,Microcin E492,Polytheonamide B,Subtilosin A,Lactococcin 972,Closticin 574,Colicin E6,Colicin E9
phylum_group,,,,,,,,,,,,,,,,,,,,
Aenigmatarchaeota,132,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0
Altiarchaeota,26,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
Asgardarchaeota,188,2,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0
Hadarchaeota,12,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
Halobacteriota,739,0,4,2,1,0,1,1,0,1,2,0,0,0,2,0,0,0,0,0
Hydrothermarchaeota,16,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
Iainarchaeota,68,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0
Methanobacteriota,207,1,0,0,0,0,0,0,0,0,0,0,0,0,0,2,0,0,0,0
Micrarchaeota,242,1,0,1,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0


In [116]:
bacterial_heatmap_df = compute_heatmap_data(
    gtdb_metadata, 
    bacteriocins_df[bacteriocins_df['domain'] == 'Bacteria'].copy(), 
    phylum_groups=bac_phylum_groups, 
    bacteriocin_names=bacteriocins_to_plot, 
)
bacterial_heatmap_df.head()

,n_assemblies,Lactococcin A,Nocardithiocin,Enterocin P,Lactococcin MMFII,Colicin M,Sakacin A,Actagardine,Aureocin A53,Bacteriocin 31,Enterocin L50b,Garvicin Q,Hiracin JM79,Microcin E492,Polytheonamide B,Subtilosin A,Lactococcin 972,Closticin 574,Colicin E6,Colicin E9
phylum_group,,,,,,,,,,,,,,,,,,,,
Acidobacteriota,1094,4,2,1,0,2,1,0,0,0,0,2,0,0,26,0,0,0,1,2
Actinomycetota,4027,11,8,2,0,4,0,7,0,0,1,0,2,0,23,1,139,143,17,11
Aquificota,84,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0
Armatimonadota,187,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0
Bacillota,10923,17,3,8,2,3,1,0,1,10,3,3,16,3,59,18,131,60,10,17


In [117]:
def make_heatmap_list(heatmap_df, amp_ids):
    heatmap_data = []
    for phylum in heatmap_df.index:
        row = [phylum]
        heatmap_row = heatmap_df.loc[phylum]
        for amp_id in amp_ids:
            c = round(100 * heatmap_row[amp_id] / heatmap_row['n_assemblies'], 3)
            row.append(c)

        heatmap_data.append(row)

    return heatmap_data

In [118]:
archaeal_heatmap_data = make_heatmap_list(archaeal_heatmap_df, bacteriocins_to_plot)

itol_heatmap_annotations(
        archaeal_heatmap_data,
        field_labels=bacteriocins_to_plot,
        output_path=outputs_folder / 'heatmap' / 'ar53_phylum_subset.heatmap.txt',
        dataset_label='% genomes in archaeal phylum',
        color_min='#ffffff',
        color_max='#f58231',
        min_value=0,
        max_value=2,
    )

In [119]:
bacterial_heatmap_data = make_heatmap_list(bacterial_heatmap_df, bacteriocins_to_plot)

itol_heatmap_annotations(
        bacterial_heatmap_data,
        field_labels=bacteriocins_to_plot,
        output_path=outputs_folder / 'heatmap' / 'bac120_phylum_subset.heatmap.txt',
        dataset_label='% genomes in bacterial phylum',
        color_min='#ffffff',
        color_max='#0173b2',
        min_value=0,
        max_value=2,
    )